In [2]:
# ============================================================
# IgGM 推理调试 Notebook
# ============================================================

# %% [1] 环境初始化
import os, sys
os.environ["HSAKMT_DEBUG_LEVEL"] = "3"
os.environ["HSA_ENABLE_INTERRUPT"] = "0"

ROOT = "/root/private_data/luog/codex/IgGM"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import yaml
from pathlib import Path
from typing import Dict, Any

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
CONFIG_PATH = "/root/private_data/luog/codex/IgGM/config/train_0507_2.yaml"
CKPT_PATH   = None  # 填入 checkpoint 路径，None 则用随机初始化

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# %% [2] 加载配置
def load_config(path: str) -> Dict[str, Any]:
    with open(path) as f:
        return yaml.safe_load(f)

cfg = load_config(CONFIG_PATH)
print(yaml.dump(cfg, default_flow_style=False))

# %% [3] 构建模型
from IgGM.model import DesignModel, esm_ppi_650m_ab
from IgGM.model.arch.core.diffuser import Diffuser
from IgGM.model.factory import build_design_model_module, build_ppi_featurizer_module
from IgGM.utils import IGSO3Buffer
from src.iggm_lightning import (
    IgGMLightningModule,
    IgGMLossConfig,
    MetricConfig,
    OptimizerConfig,
    ProcessedSabdabDataModule,
    StageTrainingConfig,
)

class DotConfig:
    def __init__(self, **kwargs): self.__dict__.update(kwargs)

# PLM featurizer
ppi_ckpt = cfg["model"].get("ppi_ckpt") or esm_ppi_650m_ab()
plm_featurizer = build_ppi_featurizer_module(ppi_ckpt)
model_cfg = DotConfig(
    c_s=getattr(plm_featurizer, "c_s", None),
    c_p=getattr(plm_featurizer, "c_z", None),
)

# DesignModel
design_ckpt = cfg["model"].get("design_ckpt") or ""
design_model = (
    build_design_model_module(design_ckpt, model_cfg)
    if design_ckpt else
    DesignModel(n_dims_sfea_init=model_cfg.c_s, n_dims_pfea_init=model_cfg.c_p)
)

# Diffuser
igso3 = None
if cfg["model"].get("igso3_buffer"):
    igso3 = IGSO3Buffer()
    igso3.load(cfg["model"]["igso3_buffer"])
diffuser = Diffuser(
    igso3_buffer=igso3,
    occupancy_mode=cfg["stage_training"].get("occupancy_mode", "joint_predict"),
)

# Lightning Module
lit_model = IgGMLightningModule(
    model=design_model,
    plm_featurizer=plm_featurizer,
    diffuser=diffuser,
    optimizer_cfg=OptimizerConfig(
        lr=cfg["optimizer"]["lr"],
        weight_decay=cfg["optimizer"]["weight_decay"],
    ),
    loss_cfg=IgGMLossConfig(vio_weight=cfg["loss"]["vio_weight"]),
    metric_cfg=MetricConfig(dockq_threshold=cfg["metrics"]["dockq_threshold"]),
    stage_cfg=StageTrainingConfig(
        stage1_epochs=cfg["stage_training"]["stage1_epochs"],
        stage2_enable_seq_recovery=cfg["stage_training"]["stage2_enable_seq_recovery"],
        stage2_mix_weights={
            "cdr_h3": cfg["stage_training"]["mix_cdr_h3"],
            "cdr_h1": cfg["stage_training"]["mix_cdr_h1"],
            "cdr_h2": cfg["stage_training"]["mix_cdr_h2"],
            "cdr_all": cfg["stage_training"]["mix_cdr_all"],
        },
    ),
)

# 加载 checkpoint（可选）
if CKPT_PATH and Path(CKPT_PATH).exists():
    state = torch.load(CKPT_PATH, map_location="cpu")
    sd = state.get("state_dict", state)
    missing, unexpected = lit_model.load_state_dict(sd, strict=False)
    print(f"Loaded ckpt: missing={len(missing)}, unexpected={len(unexpected)}")
    if missing: print("Missing keys sample:", missing[:5])
else:
    print("No checkpoint loaded, using random init")

lit_model = lit_model.to(DEVICE)
lit_model.eval()
print("Model ready")

# %% [4] 数据加载
dm = ProcessedSabdabDataModule(
    metadata_path=cfg["data"]["metadata"],
    pdb_dir=cfg["data"]["pdb_dir"],
    batch_size=1,
    num_workers=0,
    train_ids_path=cfg["data"].get("train_ids") or None,
    val_ids_path=cfg["data"].get("val_ids") or None,
    test_ids_path=cfg["data"].get("test_ids") or None,
    train_cluster_path=cfg["data"].get("train_clusters") or None,
    samples_dir=cfg["data"].get("samples_dir") or None,
    n_steps=cfg["data"]["n_steps"],
    lazy_cache_size=cfg["stage_training"].get("lazy_cache_size", 128),
    forward_chunk_size=cfg["data"].get("forward_chunk_size") or None,
    max_antigen_len=cfg["data"].get("max_antigen_len") or None,
)
dm.setup()

# 取一个 val batch
val_loader = dm.val_dataloader()
batch = next(iter(val_loader))

def move_to_device(obj, device):
    if torch.is_tensor(obj): return obj.to(device)
    if isinstance(obj, dict): return {k: move_to_device(v, device) for k, v in obj.items()}
    if isinstance(obj, list): return [move_to_device(v, device) for v in obj]
    if isinstance(obj, tuple): return tuple(move_to_device(v, device) for v in obj)
    return obj

batch = move_to_device(batch, DEVICE)
print("Batch keys:", list(batch.keys()))
payload = batch["payload"]
print("Payload keys:", list(payload.keys()))

# %% [5] 单步前向推理
idx_step = int(batch["idx_step"])
prot_data_curr = lit_model._move_to_device(payload["prot_data_curr"])

with torch.no_grad():
    inputs  = lit_model._build_inputs_cm(prot_data_curr, idx_step)
    outputs = lit_model.model(inputs, chunk_size=batch.get("chunk_size"))

print("Inputs keys:", list(inputs.keys()))
print("Outputs 3d keys:", list(outputs["3d"].keys()))
print("pred cord shape:", outputs["3d"]["cord"][-1].shape)

# %% [6] 损失计算与分项打印
with torch.no_grad():
    loss_dict = lit_model.loss_fn(inputs, outputs)

print("\n===== Loss =====")
for k, v in loss_dict.items():
    val = v.item() if torch.is_tensor(v) else v
    print(f"  {k:25s}: {val:.6f}")

# %% [7] 虚拟原子精度分析（核心调试）
from src.iggm_lightning.atom14_sync import Atom14SeqSync

atom14_sync = Atom14SeqSync()
true_seq  = payload.get("seq_true", inputs["seq-o"][0])
pred_cord = outputs["3d"]["cord"][-1][0]   # [L, 14, 3]
tgt_cord  = inputs.get("cords_atom14", inputs["cord-o"])
if tgt_cord.ndim == 4: tgt_cord = tgt_cord[0]

cdr_mask  = inputs["cdr_mask"]
if cdr_mask.ndim == 2: cdr_mask = cdr_mask[0]
cdr_mask_bool = cdr_mask.to(torch.bool)

# n_real 查表
from IgGM.protein.prot_constants import ATOM_NAMES_PER_RESD, RESD_MAP_1TO3, RESD_NAMES_1C
n_real_dict = {
    aa: len(ATOM_NAMES_PER_RESD[RESD_MAP_1TO3[aa]])
    for aa in RESD_NAMES_1C
}

print("\n===== 虚拟原子精度（CDR区域）=====")
print(f"{'残基':>6} {'AA':>4} {'slot':>5} {'d_to_N':>8} {'d_to_O':>8} {'归属':>6} {'误差':>8}")
print("-" * 55)

total_virt, correct_virt = 0, 0
dist_list = []

for ridx, aa in enumerate(true_seq):
    if ridx >= len(cdr_mask_bool) or not bool(cdr_mask_bool[ridx]):
        continue
    if aa not in n_real_dict:
        continue

    n_real = n_real_dict[aa]
    if n_real >= 14:
        continue  # W 等无虚拟原子

    N_pos = pred_cord[ridx, 0]
    O_pos = pred_cord[ridx, 3]

    for slot in range(n_real, 14):
        pred_atom = pred_cord[ridx, slot]
        gt_atom   = tgt_cord[ridx, slot]

        d_n   = float(torch.norm(pred_atom - N_pos).item())
        d_o   = float(torch.norm(pred_atom - O_pos).item())
        d_gt  = float(torch.norm(pred_atom - gt_atom).item())
        belong = "N" if d_n <= d_o else "O"
        min_d  = min(d_n, d_o)

        dist_list.append(min_d)
        total_virt += 1
        if min_d < 0.5:
            correct_virt += 1

        print(f"{ridx:6d} {aa:>4} {slot:>5d} {d_n:8.3f} {d_o:8.3f} {belong:>6} {d_gt:8.3f}Å")

print("-" * 55)
if total_virt > 0:
    print(f"虚拟原子总数: {total_virt}")
    print(f"距N/O < 0.5Å: {correct_virt} ({100*correct_virt/total_virt:.1f}%)")
    print(f"平均最近距离: {np.mean(dist_list):.3f}Å")
    print(f"中位最近距离: {np.median(dist_list):.3f}Å")
    print(f"距N/O < 1.0Å: {sum(d<1.0 for d in dist_list)} ({100*sum(d<1.0 for d in dist_list)/total_virt:.1f}%)")

# %% [8] 序列解码与 AAR 计算
pred_seq = atom14_sync.decode_cdr_sequence(
    seq_true=true_seq,
    pred_cord_n14_tf=pred_cord,
    pred_cmsk_n14_tf=inputs.get("cmsk_atom14", inputs["cmsk-p"])[0],
    cdr_mask=inputs["cdr_mask"],
)

print("\n===== 序列对比（CDR区域）=====")
print(f"True: {true_seq}")
print(f"Pred: {pred_seq}")
print()

cdr_correct = cdr_total = 0
for i, (p, t) in enumerate(zip(pred_seq, true_seq)):
    if i < len(cdr_mask_bool) and bool(cdr_mask_bool[i]):
        cdr_total += 1
        if p == t: cdr_correct += 1
        marker = "✓" if p == t else f"✗({t}→{p})"
        print(f"  pos {i:3d}: pred={p}  true={t}  {marker}")

if cdr_total > 0:
    print(f"\nCDR AAR: {cdr_correct}/{cdr_total} = {100*cdr_correct/cdr_total:.1f}%")

# %% [9] 不同阈值下的序列恢复率测试
print("\n===== 阈值敏感性分析 =====")
for thresh in [0.3, 0.5, 0.8, 1.0, 1.2, 1.5]:
    tmp_sync = Atom14SeqSync(decode_threshold=thresh)
    tmp_seq = tmp_sync.decode_cdr_sequence(
        seq_true=true_seq,
        pred_cord_n14_tf=pred_cord,
        pred_cmsk_n14_tf=inputs.get("cmsk_atom14", inputs["cmsk-p"])[0],
        cdr_mask=inputs["cdr_mask"],
    )
    correct = sum(p == t for i, (p, t) in enumerate(zip(tmp_seq, true_seq))
                  if i < len(cdr_mask_bool) and bool(cdr_mask_bool[i]))
    total = sum(1 for i in range(len(true_seq))
                if i < len(cdr_mask_bool) and bool(cdr_mask_bool[i]))
    print(f"  thresh={thresh:.1f}Å  AAR={correct}/{total}={100*correct/max(total,1):.1f}%")

# %% [10] 完整评估指标
tgt_cord_metric = inputs["cord-o"]
if tgt_cord_metric.ndim == 4: tgt_cord_metric = tgt_cord_metric[0]

with torch.no_grad():
    metric_dict = lit_model.metric_fn(
        pred_cord,
        tgt_cord_metric,
        pred_seq,
        true_seq,
        (payload.get("cdr_sequences") or {}).get("cdr_H3", []),
        asym_id=inputs.get("asym-id"),
        cdr_sequences=(payload.get("cdr_sequences") or {}),
        seq_lengths=(payload.get("sequence_lengths") or {}),
    )

print("\n===== 结构评估指标 =====")
for k, v in metric_dict.items():
    val = v.item() if torch.is_tensor(v) else v
    print(f"  {k:25s}: {val:.4f}")

# %% [11] 多步去噪轨迹分析（观察虚拟原子收敛过程）
print("\n===== 去噪轨迹：虚拟原子收敛过程 =====")
n_steps_total = cfg["data"]["n_steps"]
probe_steps = [int(n_steps_total * r) for r in [0.9, 0.7, 0.5, 0.3, 0.1, 0.0]]
probe_steps = sorted(set(max(0, min(s, n_steps_total-1)) for s in probe_steps))

results = []
for step in probe_steps:
    with torch.no_grad():
        inp = lit_model._build_inputs_cm(prot_data_curr, step)
        out = lit_model.model(inp, chunk_size=batch.get("chunk_size"))
    pc = out["3d"]["cord"][-1][0]

    # 统计虚拟原子精度
    dists = []
    for ridx, aa in enumerate(true_seq):
        if ridx >= len(cdr_mask_bool) or not bool(cdr_mask_bool[ridx]): continue
        if aa not in n_real_dict: continue
        nr = n_real_dict[aa]
        if nr >= 14: continue
        N_pos = pc[ridx, 0]; O_pos = pc[ridx, 3]
        for slot in range(nr, 14):
            d_n = float(torch.norm(pc[ridx, slot] - N_pos).item())
            d_o = float(torch.norm(pc[ridx, slot] - O_pos).item())
            dists.append(min(d_n, d_o))

    if dists:
        t_val = float(inp["sigama_t"]["cord_scale"]) if "sigama_t" in inp else float(step)/n_steps_total
        mean_d = np.mean(dists)
        frac_05 = sum(d < 0.5 for d in dists) / len(dists)
        results.append((step, t_val, mean_d, frac_05))
        print(f"  step={step:3d}  mean_dist={mean_d:.3f}Å  frac<0.5Å={100*frac_05:.1f}%")

# %% [12] 保存调试快照
import time
snap_path = f"/root/private_data/luog/codex/IgGM/see/debug_{int(time.time())}.pt"
Path(snap_path).parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "pred_cord":   pred_cord.cpu(),
    "tgt_cord":    tgt_cord.cpu(),
    "pred_seq":    pred_seq,
    "true_seq":    true_seq,
    "cdr_mask":    cdr_mask_bool.cpu(),
    "loss_dict":   {k: v.item() if torch.is_tensor(v) else v for k, v in loss_dict.items()},
    "metric_dict": {k: v.item() if torch.is_tensor(v) else v for k, v in metric_dict.items()},
    "dist_list":   dist_list,
}, snap_path)
print(f"\n调试快照已保存: {snap_path}")

Device: cuda:0
PyTorch: 2.5.1
data:
  batch_size: 1
  forward_chunk_size: 128
  max_antigen_len: 256
  metadata: data/sabdab_2603/processed/sabdab_2603/metadata.json
  n_steps: 200
  num_workers: 1
  pdb_dir: data/sabdab_2603/processed/pdb
  samples_dir: data/sabdab_2603/processed/sabdab_2603/samples
  test_ids: data/sabdab_2603/processed/split_onetest/test_prot_ids.txt
  train_clusters: data/sabdab_2603/processed/split_onetest/train_prot_cluster.txt
  train_ids: data/sabdab_2603/processed/split_onetest/train_prot_ids.txt
  val_ids: data/sabdab_2603/processed/split_onetest/val_prot_ids.txt
loss:
  vio_weight: 0.02
metrics:
  dockq_threshold: 0.23
model:
  design_ckpt: ''
  igso3_buffer: /root/private_data/luog/codex/IgGM/checkpoints/igso3_buffer.pth
  ppi_ckpt: /root/private_data/luog/codex/IgGM/checkpoints/esm_ppi_650m_ab.pth
optimizer:
  grad_clip: 1.0
  lr: 0.0005
  weight_decay: 0.01
runtime:
  output_dir: outputs/lightning_train050902
  resume: false
  run_test: true
  seed: 56
sc

ValueError: Expected [L,14], got (14,)

In [1]:
from IgGM.utils import (
    IsotropicGaussianSO3,
    extract_clean_fr_reference,
    extract_per_loop_clean_local_coords,
    global_to_local_coords,
    local_to_global_coords,
    merge_noisy_fr_and_loops,
    prob2seq,
    ptr2ss,
    rebuild_loops_from_local_coords,
    so3_scale,
    ss2ptr,
    skew2vec,
    log_rmat,
)
import torch

def get_valid_rotation_matrix(batch_size=1):
    # 1. 随机生成一个 3x3 的矩阵
    rand_mat = torch.randn(batch_size, 3, 3)
    
    # 2. 进行 QR 分解，Q 是正交矩阵
    q, r = torch.linalg.qr(rand_mat)
    
    # 3. 确保行列式为 1 (防止生成的是镜像矩阵)
    # 如果行列式为 -1，只需将 Q 的第一列取反即可
    det = torch.linalg.det(q)
    mask = (det < 0).float().view(-1, 1, 1)
    q = q.clone() # 避免原地修改
    q[:, :, 0] = q[:, :, 0] * (1 - 2 * mask) # 如果 det < 0，乘以 -1
    
    return q

# --- 你的测试代码 ---
test_coords = torch.randn(1, 100, 3)
test_R = get_valid_rotation_matrix(batch_size=1)  # 生成合法的旋转矩阵 形状为 [1, 3, 3]
test_T = torch.randn(1, 3)

# 打印检查旋转矩阵是否合法
# 1. R * R^T 应该等于单位阵
print("正交性检查 (应接近0):", torch.max(torch.abs(torch.bmm(test_R, test_R.transpose(1, 2)) - torch.eye(3))))
# 2. 行列式应该等于 1
print("行列式检查 (应接近1):", torch.linalg.det(test_R).item())


# # 伪代码：测试转换是否完美互逆
# test_coords = torch.randn(1, 100, 3)
# test_R = 随机生成一个合法的旋转矩阵
# test_T = torch.randn(1, 3)

local = global_to_local_coords(test_coords, test_R, test_T)
recon = local_to_global_coords(local, test_R, test_T)

# 这必须输出 True！如果输出 False，或者误差很大，说明 Bug 就在这两个函数里！
print(torch.allclose(test_coords, recon, atol=1e-5))

[2026-05-26 11:37:27,135] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


正交性检查 (应接近0): tensor(1.1921e-07)
行列式检查 (应接近1): 0.9999999403953552
True


In [ ]:
#!/usr/bin/env python3

import argparse
import json
import math

from DockQ.DockQ import load_PDB, run_on_all_native_interfaces


def parse_chain_list(text: str):
    return [x.strip() for x in text.split(",") if x.strip()]


def finite(value):
    return math.isfinite(float(value))


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--native", required=True)
    parser.add_argument(
        "--model-chains",
        required=True,
        help="Comma-separated model chain IDs, e.g. H,L,A",
    )
    parser.add_argument(
        "--native-chains",
        required=True,
        help="Comma-separated native chain IDs, e.g. H,L,A",
    )
    parser.add_argument(
        "--antibody-chains",
        default="H,L",
        help="Comma-separated antibody chain IDs in native structure",
    )
    parser.add_argument(
        "--antigen-chain",
        default="A",
        help="Antigen chain ID in native structure",
    )
    args = parser.parse_args()

    model_chain_ids = parse_chain_list(args.model_chains)
    native_chain_ids = parse_chain_list(args.native_chains)
    antibody_chain_ids = set(parse_chain_list(args.antibody_chains))
    antigen_chain_id = args.antigen_chain

    if len(model_chain_ids) != len(native_chain_ids):
        raise ValueError(
            "model/native chain list lengths must match"
        )

    model_structure = load_PDB(args.model)
    native_structure = load_PDB(args.native)

    model_available = {chain.id for chain in model_structure}
    native_available = {chain.id for chain in native_structure}

    missing_model = set(model_chain_ids) - model_available
    missing_native = set(native_chain_ids) - native_available

    if missing_model:
        raise ValueError(
            f"model is missing requested chains: {sorted(missing_model)}"
        )

    if missing_native:
        raise ValueError(
            f"native is missing requested chains: {sorted(missing_native)}"
        )

    if antigen_chain_id not in native_chain_ids:
        raise ValueError(
            f"antigen chain {antigen_chain_id} is not in native chain list"
        )

    native_to_model_chain_map = dict(
        zip(native_chain_ids, model_chain_ids)
    )

    result = run_on_all_native_interfaces(
        model_structure,
        native_structure,
        chain_map=native_to_model_chain_map,
    )

    if not isinstance(result, tuple) or len(result) != 2:
        raise RuntimeError(
            "Unexpected DockQ API return value. "
            "This script intentionally does not use a fallback."
        )

    interface_results, total_score = result

    if not isinstance(interface_results, dict):
        raise RuntimeError(
            "DockQ interface result is not a dictionary"
        )

    if not finite(total_score):
        raise RuntimeError("DockQ total score is not finite")

    selected = {}

    for pair_name, metrics in interface_results.items():
        if not isinstance(metrics, dict):
            raise RuntimeError(
                f"Invalid metrics for interface {pair_name}"
            )

        native_pair = set(pair_name)

        is_antibody_antigen = (
            antigen_chain_id in native_pair
            and bool(native_pair & antibody_chain_ids)
        )

        if not is_antibody_antigen:
            continue

        required = (
            "DockQ",
            "iRMSD",
            "LRMSD",
            "fnat",
            "clashes",
        )

        missing = [key for key in required if key not in metrics]
        if missing:
            raise RuntimeError(
                f"DockQ interface {pair_name} missing keys: {missing}"
            )

        for key in ("DockQ", "iRMSD", "LRMSD", "fnat"):
            if not finite(metrics[key]):
                raise RuntimeError(
                    f"DockQ interface {pair_name} has non-finite {key}"
                )

        selected[pair_name] = {
            "DockQ": float(metrics["DockQ"]),
            "iRMSD": float(metrics["iRMSD"]),
            "LRMSD": float(metrics["LRMSD"]),
            "fnat": float(metrics["fnat"]),
            "clashes": int(metrics["clashes"]),
            "chain1": metrics.get("chain1"),
            "chain2": metrics.get("chain2"),
        }

    if not selected:
        raise RuntimeError(
            "No antibody-antigen interfaces were returned by native DockQ"
        )

    mean_dockq = sum(
        item["DockQ"] for item in selected.values()
    ) / len(selected)

    output = {
        "model": args.model,
        "native": args.native,
        "native_to_model_chain_map": native_to_model_chain_map,
        "dockq_total_returned_by_api": float(total_score),
        "selected_antibody_antigen_interfaces": selected,
        "mean_antibody_antigen_dockq": mean_dockq,
    }

    print(json.dumps(output, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()